# Phase 5 - Legal Statute Identification (lightweight Phase-1 model)

**Task:** multi-label classification - for each legal document predict the set of relevant statute IDs; primary metric macro-F1 (rare statutes matter), secondary micro-F1 and subset accuracy.

**What you need on disk (nothing else - the model code is embedded below):**
1. your Phase-1 lightweight checkpoint `lightweight_final.pt` (must contain `model_state` and `config`)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` (the `.pt` does **not** contain it)
3. a Hugging Face token (Kaggle secret `hf_token`)

**Approach (hierarchical, long-document):** a document is cut into `CH`-token chunks; up to `K_CHUNKS` are selected (`SELECT`); the lightweight LM (4 shared blocks run twice, causal 128-token local attention) encodes each chunk into a vector; attention pooling (or mean / BiGRU) combines the chunk vectors; a linear layer gives one logit per statute (sigmoid, independent = binary relevance). The head bias starts at the label prior. Losses: BCE, class-weighted BCE (default), focal, asymmetric. Decoding: global dev-tuned threshold, optional per-statute thresholds, and every document always gets at least its top statute.

**Experiments:** full fine-tuning, partial, LoRA, **LoRMA+** (compulsory), bottleneck adapters, knowledge distillation (student = half the unique blocks), then loss / long-document / aggregator ablations, a rare-statute analysis and an ensemble. AdaLoRA, QLoRA and prefix tuning are **not** included.

> The course requires all **four** Phase-1 architectures and >= 6 substantially different experiments (>= 5 different adaptation techniques, LoRMA and knowledge distillation among them). The training budgets (`minutes=`) and learning rates are placeholders - choosing and justifying them is your job (course rules). Training is time-budgeted: each run trains for `minutes` of wall-clock time, so it is usually less than one epoch - check the `epochs` figure printed after each run.

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/YOUR_DATASET/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/YOUR_DATASET/lightweight_final.pt"       # <-- Phase-1 lightweight checkpoint
HF_REPO = "Exploration-Lab/CS-781-Capstone"
OUT = "/kaggle/working"

CH         = 512      # tokens per chunk (<= the lightweight model's max_seq_len of 1024)
STRIDE_CH  = 512      # chunk stride; < CH gives overlapping chunks
K_CHUNKS   = 4        # at most this many chunks of a document go through the model (cost is linear in this; 8 was ~2x slower)
SELECT     = "headtail"   # which chunks: head | tail | headtail | spread   (the long-document / section experiment)
CAP_CHARS  = 60000    # very long judgments: keep this many characters from each end (bounds tokenisation time)

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

from collections import Counter

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("hf_token")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 lightweight model code (copied unchanged) and checkpoint loading

In [ ]:
# ---- lightweight_layers.py (Phase 1, unchanged) ----
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight.to(x.dtype)


def get_alibi_slopes(n_heads):
    def power_of_2_slopes(n):
        start = 2 ** (-2 ** -(math.log2(n) - 3))
        ratio = start
        return torch.tensor([start * ratio ** i for i in range(n)], dtype=torch.float32)
    if math.log2(n_heads).is_integer():
        return power_of_2_slopes(n_heads)
    closest_power = 2 ** int(math.floor(math.log2(n_heads)))
    extra = n_heads - closest_power
    slopes = power_of_2_slopes(closest_power)
    extra_slopes = get_alibi_slopes(2 * closest_power)[0::2]
    return torch.cat([slopes, extra_slopes[:extra]])


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class LocalGQAAttention(nn.Module):
    """Causal sliding-window grouped-query attention with ALiBi."""
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm=True):
        super().__init__()
        assert d_model % n_q_heads == 0 and n_q_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_q_heads, self.n_kv_heads = n_q_heads, n_kv_heads
        self.groups = n_q_heads // n_kv_heads
        self.d_head = d_model // n_q_heads
        self.window_size = min(window_size, max_seq_len)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)
        self.dropout = dropout
        self.register_buffer("alibi_slopes", get_alibi_slopes(n_q_heads), persistent=False)
        distance = torch.arange(self.window_size - 1, -1, -1, dtype=torch.float32)
        self.register_buffer("distance", distance, persistent=False)
        query_pos = torch.arange(max_seq_len).unsqueeze(1)
        window_pos = torch.arange(self.window_size).unsqueeze(0)
        valid = window_pos >= (self.window_size - 1 - query_pos)
        self.register_buffer("valid_mask", valid, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        q = q.view(B, self.n_kv_heads, self.groups, T, self.d_head)
        W = self.window_size
        k_win = F.pad(k, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)   # (B, Hkv, T, W, D)
        v_win = F.pad(v, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)
        scores = torch.einsum("bhgtd,bhtwd->bhgtw", q, k_win) / math.sqrt(self.d_head)
        slopes = self.alibi_slopes.view(1, self.n_kv_heads, self.groups, 1, 1)
        scores = scores - slopes * self.distance.view(1, 1, 1, 1, W)
        mask = self.valid_mask[:T, :W].view(1, 1, 1, T, W)
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = F.softmax(scores.float(), dim=-1).to(scores.dtype)
        if self.training and self.dropout > 0:
            attn = F.dropout(attn, p=self.dropout)
        out = torch.einsum("bhgtw,bhtwd->bhgtd", attn, v_win)
        out = out.reshape(B, self.n_q_heads, T, self.d_head).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, d_ff, dropout, qk_norm):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = LocalGQAAttention(d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm)
        self.ln2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


# ---- model_arch_lightweight.py (Phase 1, unchanged) ----
from dataclasses import dataclass

@dataclass
class LightweightConfig:
    vocab_size: int
    d_model: int = 1024
    n_q_heads: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 1024
    window_size: int = 128
    dropout: float = 0.0
    qk_norm: bool = True
    n_unique_blocks: int = 4       # 4 unique blocks ...
    n_repeats: int = 2             # ... each executed twice: B1 B2 B3 B4 B1 B2 B3 B4


class LightweightLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            LightweightBlock(cfg.d_model, cfg.n_q_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.window_size,
                             cfg.d_ff, cfg.dropout, cfg.qk_norm)
            for _ in range(cfg.n_unique_blocks)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.tok_emb(x)
        for _ in range(self.cfg.n_repeats):
            for block in self.blocks:
                x = block(x)
        return self.lm_head(self.ln_f(x))

In [ ]:
def load_lm(path=CKPT_PATH, dropout=0.1):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = LightweightConfig(**{k: v for k, v in ck["config"].items() if k in LightweightConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    cfg.dropout = dropout                                    # Phase 1 used 0.0; a little helps fine-tuning
    lm = LightweightLM(cfg)
    lm.load_state_dict(ck["model_state"])                    # alibi / mask buffers are non-persistent: rebuilt, not loaded
    return lm

_lm = load_lm(dropout=0.0)
print(_lm.cfg)
print(f"loaded Phase-1 lightweight LM: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M unique params "
      f"({_lm.cfg.n_unique_blocks} blocks x {_lm.cfg.n_repeats} repeats)")
assert _lm.cfg.max_seq_len >= CH, "CH must be <= max_seq_len"

## 4. Data
### 4.1 Peek at the raw JSON (verify the layout and the label key before trusting the parser)

In [ ]:
def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

for name, path in [("train", "Phase_5_LSI/Train/train.json"), ("dev", "Phase_5_LSI/Train/dev.json"),
                   ("test", "Phase_5_LSI/Test/test.json")]:
    raw = hf_json(path)
    first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
    if isinstance(first, dict):
        show = {k: (str(v)[:120] + "..." if len(str(v)) > 120 else v) for k, v in first.items()}
    else:
        show = str(first)[:300]
    print(name, type(raw).__name__, "len", len(raw), "\n   first:", show, "\n")

### 4.2 Parse train / dev / test, build the statute label space

In [ ]:
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "y")

def parse_docs(raw):
    if isinstance(raw, dict):
        items = [(k, v) for k, v in raw.items()]
    else:
        items = [(d["id"], d) for d in raw]
    docs = []
    for did, d in items:
        if not isinstance(d, dict):
            d = {"text": d}
        lab = next((d[k] for k in LABEL_KEYS if k in d), None)
        if lab is not None:
            lab = sorted({int(x) for x in (lab if isinstance(lab, (list, tuple)) else [lab])})
        docs.append(dict(id=d.get("id", did), text=d["text"], labels=lab))
    return docs

train_docs = parse_docs(hf_json("Phase_5_LSI/Train/train.json"))
dev_docs   = parse_docs(hf_json("Phase_5_LSI/Train/dev.json"))
test_docs  = parse_docs(hf_json("Phase_5_LSI/Test/test.json"))
assert all(d["labels"] is not None for d in train_docs + dev_docs), "labels key not found - check the PEEK output above"
assert len({str(d["id"]) for d in test_docs}) == len(test_docs), "duplicate test ids"

# label space = every statute id seen in train or dev (the evaluator derives the label dimension from the reference data)
LABELS = sorted({l for d in train_docs + dev_docs for l in d["labels"]})
L2I = {l: i for i, l in enumerate(LABELS)}
NL = len(LABELS)
for d in train_docs + dev_docs + test_docs:
    y = np.zeros(NL, dtype=np.float32)
    for l in (d["labels"] or []):
        y[L2I[l]] = 1.0
    d["y"] = y

Ytr = np.stack([d["y"] for d in train_docs]); Ydv = np.stack([d["y"] for d in dev_docs])
freq = Ytr.sum(0)
PRIOR_LOGIT = np.log((freq + 1) / (len(train_docs) + 2 - freq - 1))          # head bias init: start from the label prior
print(f"docs: train {len(train_docs)} | dev {len(dev_docs)} | test {len(test_docs)} | statutes (label dim) {NL}")
print(f"labels/doc: train mean {Ytr.sum(1).mean():.2f}, max {int(Ytr.sum(1).max())} | docs with no label: {int((Ytr.sum(1)==0).sum())}")
print(f"statutes with < 5 train docs: {int((freq < 5).sum())} | with 0 train docs: {int((freq == 0).sum())} | most frequent covers {freq.max()/len(train_docs):.0%} of docs")
print("median chars/doc:", int(np.median([len(d["text"]) for d in train_docs])), "| max:", max(len(d["text"]) for d in train_docs))

### 4.3 Tokenise and chunk

In [ ]:
def chunk_ids(ids, ch=CH, stride=STRIDE_CH):
    starts = [0]
    while starts[-1] + ch < len(ids):
        starts.append(starts[-1] + stride)
    return [ids[s:s + ch] for s in starts]

def tokenise_doc(text):
    if len(text) > 2 * CAP_CHARS:
        text = text[:CAP_CHARS] + " " + text[-CAP_CHARS:]
    ids = tokenizer.encode(text)
    return ids if ids else [tokenizer.eos_id]

t0 = time.time()
for d in train_docs + dev_docs + test_docs:
    ids = tokenise_doc(d["text"])
    d["n_tok"] = len(ids)
    d["chunks"] = chunk_ids(ids)
print(f"tokenised in {time.time()-t0:.0f}s | tokens/doc: median {int(np.median([d['n_tok'] for d in train_docs]))}, "
      f"p90 {int(np.percentile([d['n_tok'] for d in train_docs], 90))} | chunks/doc: "
      f"median {int(np.median([len(d['chunks']) for d in train_docs]))}, max {max(len(d['chunks']) for d in train_docs)}")

### 4.4 Chunk selection and batching

In [ ]:
def select_idx(n, select=None, K=None):
    """Which of the n chunks of a document are used.  head = first K, tail = last K (judgments end with the outcome),
    headtail = first K/2 + last K/2, spread = K evenly spaced chunks over the whole document."""
    select, K = select or SELECT, K or K_CHUNKS
    if n <= K:
        return list(range(n))
    if select == "head":
        return list(range(K))
    if select == "tail":
        return list(range(n - K, n))
    if select == "headtail":
        h = K // 2
        return list(range(h)) + list(range(n - (K - h), n))
    if select == "spread":
        return sorted({round(i * (n - 1) / max(1, K - 1)) for i in range(K)})
    raise ValueError(select)

def collate(docs, select=None, K=None):
    """-> X (N,T) chunk tokens | lens (N,) | cnt (D,) chunks per doc | Y (D, NL) multi-hot labels.  Chunks of all docs are stacked."""
    seqs, cnt = [], []
    for d in docs:
        idx = select_idx(len(d["chunks"]), select, K)
        seqs += [d["chunks"][i] for i in idx]; cnt.append(len(idx))
    T = max(len(s) for s in seqs)
    X = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)
    lens = torch.tensor([len(s) for s in seqs])
    Y = torch.tensor(np.stack([d["y"] for d in docs]))
    return X, lens, torch.tensor(cnt), Y

print("example selections for a 20-chunk doc:", {s: select_idx(20, s) for s in ("head", "tail", "headtail", "spread")})

### 4.5 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? LM loss on real legal text should be ~3-4 nats; ~10 means a wrong tokenizer pkl.
x = torch.tensor([train_docs[0]["chunks"][0]]); _lm.eval()
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on a train chunk:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3), "| tokens:", x.size(1))
del _lm

## 5. Metrics (macro-F1 / micro-F1 / subset accuracy), thresholds, multi-label losses

In [ ]:
GRID = np.round(np.arange(0.02, 0.92, 0.02), 2)

def label_f1(Y, P):
    """Per-statute F1 for 0/1 arrays of shape (docs, NL)."""
    tp = (Y * P).sum(0); fp = ((1 - Y) * P).sum(0); fn = (Y * (1 - P)).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0), tp, fp, fn

def multilabel_scores(Y, P):
    """Macro-F1 (primary; averaged over statutes that occur in Y), micro-F1, subset accuracy (secondary)."""
    f1, tp, fp, fn = label_f1(Y, P)
    present = Y.sum(0) > 0
    return dict(macro_f1=float(f1[present].mean()), macro_f1_all=float(f1.mean()),
                micro_f1=float(2 * tp.sum() / max(1.0, 2 * tp.sum() + fp.sum() + fn.sum())),
                subset_acc=float((Y == P).all(1).mean()), avg_pred=float(P.sum(1).mean()))

def decode(probs, thr, top1=True):
    """probs (D, NL) -> 0/1 matrix. thr: one global float or a per-statute array. top1: a document with no statute above the
    threshold still gets its single most likely statute (documents always have >= 1 label)."""
    P = (probs >= thr).astype(np.float32)
    if top1:
        empty = P.sum(1) == 0
        P[empty, probs[empty].argmax(1)] = 1.0
    return P

def best_global_threshold(probs, Y):
    best = max(GRID, key=lambda t: multilabel_scores(Y, decode(probs, t))["macro_f1"])
    return float(best), multilabel_scores(Y, decode(probs, best))["macro_f1"]

def per_label_thresholds(probs, Y, glob, min_pos=5):
    """One threshold per statute, fitted on dev; statutes with fewer than min_pos positives keep the global threshold."""
    f1_by_t = np.stack([label_f1(Y, (probs >= t).astype(np.float32))[0] for t in GRID])        # (grid, NL)
    thr = np.full(Y.shape[1], glob, dtype=np.float32)
    ok = Y.sum(0) >= min_pos
    thr[ok] = GRID[f1_by_t[:, ok].argmax(0)]
    return thr

def per_label_cv(probs, Y, glob, min_pos=3):
    """Honest estimate of per-label thresholds: fit on half of dev, score on the other half (and vice versa)."""
    idx = np.arange(len(Y)); halves = [(idx % 2 == 0, idx % 2 == 1), (idx % 2 == 1, idx % 2 == 0)]
    pl, gl = [], []
    for fit, ev in halves:
        thr = per_label_thresholds(probs[fit], Y[fit], glob, min_pos)
        pl.append(multilabel_scores(Y[ev], decode(probs[ev], thr))["macro_f1"])
        gl.append(multilabel_scores(Y[ev], decode(probs[ev], glob))["macro_f1"])
    return float(np.mean(pl)), float(np.mean(gl))

def multilabel_loss(logits, Y, loss_type="wbce", pos_w=None, gamma=2.0):
    """bce | wbce (positives up-weighted per statute) | focal | asl (asymmetric loss: down-weights easy negatives).
    Summed over statutes, averaged over documents."""
    if loss_type == "bce":
        l = F.binary_cross_entropy_with_logits(logits, Y, reduction="none")
    elif loss_type == "wbce":
        l = F.binary_cross_entropy_with_logits(logits, Y, pos_weight=pos_w, reduction="none")
    elif loss_type == "focal":
        ce = F.binary_cross_entropy_with_logits(logits, Y, reduction="none")
        p = torch.sigmoid(logits); pt = p * Y + (1 - p) * (1 - Y)
        l = (1 - pt) ** gamma * ce
    elif loss_type == "asl":
        p = torch.sigmoid(logits); pn = (p - 0.05).clamp(min=0)
        l = -(Y * torch.log(p.clamp(min=1e-8)) + (1 - Y) * (pn ** 4) * torch.log((1 - pn).clamp(min=1e-8)))
    else:
        raise ValueError(loss_type)
    return l.sum(1).mean()

## 6. Adaptation methods: LoRA, LoRMA+, bottleneck adapters
The 4 unique blocks are run twice, so their adapters are shared across both passes.

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: W x + s * B A x with W frozen.  rs=True -> rank-stabilised scaling s = alpha / sqrt(r) (rsLoRA)."""
    def __init__(self, base, r=16, alpha=32, rs=False, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32, rs=False):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha, rs))
        else:
            apply_lora(child, r, alpha, rs)


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi 2025): multiplicative update  h = (I + (alpha/r) B A) W0 x ,  B=0 at init."""
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * ((z @ self.A.T) @ self.B.T)
        return z + self.base.bias if self.base.bias is not None else z

def apply_lorma(module, targets=("q_proj", "v_proj"), r=8, alpha=8):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRMALinear(child, r, alpha))
        else:
            apply_lorma(child, targets, r, alpha)



class AdapterBlock(nn.Module):
    """Bottleneck adapter (Houlsby-style) after a whole block: x + up(GELU(down(x))). `up` starts at zero = identity."""
    def __init__(self, block, d_model, bottleneck=64):
        super().__init__()
        self.block = block
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        x = self.block(x)
        return x + self.up(F.gelu(self.down(x)))

def apply_adapters(lm, bottleneck=64):
    lm.blocks = nn.ModuleList([AdapterBlock(b, lm.cfg.d_model, bottleneck) for b in lm.blocks])

def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

## 7. Model, `build_model`, distillation student

In [ ]:
from torch.utils.checkpoint import checkpoint

class CJModel(nn.Module):
    """Phase-1 lightweight LM (4 shared blocks x 2 repeats, causal 128-token local attention) reads each chunk -> chunk vector [mean ; last-token state]
    -> document-level aggregation over the chunks (agg = 'attn' learned attention pooling | 'mean' | 'gru' BiGRU)
    -> one logit per statute (multi-label)."""
    def __init__(self, lm, agg="attn", proj=512, rnn_hidden=256, dropout=0.1):
        super().__init__()
        self.lm, self.agg = lm, agg
        self.n_nograd = 0        # first n blocks (and the embedding) run under no_grad: no backward through them (set by train_fast)
        self.use_ckpt = False    # gradient checkpointing on the trainable blocks: less memory, ~30% slower
        d = lm.cfg.d_model
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d, proj), nn.GELU())
        if agg == "attn":
            self.att = nn.Linear(proj, 1); zdim = proj
        elif agg == "gru":
            self.rnn = nn.GRU(proj, rnn_hidden, batch_first=True, bidirectional=True); zdim = 2 * rnn_hidden
        elif agg == "mean":
            zdim = proj
        else:
            raise ValueError(agg)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(zdim, NL)

    def set_top_blocks(self, top):
        """Train only the last `top` block passes (4 unique blocks x 2 repeats = 8 passes): the embedding and the earlier passes
        run under no_grad. The unique blocks are shared by all passes, so their weights still train through the later passes."""
        self.n_nograd = max(0, len(self.lm.blocks) * self.lm.cfg.n_repeats - top)
        for p in self.lm.tok_emb.parameters(): p.requires_grad = False

    def head_params(self):
        mods = [self.proj, self.out] + [getattr(self, n) for n in ("att", "rnn") if hasattr(self, n)]
        return [p for m in mods for p in m.parameters()]

    def features(self, x):
        passes = [blk for _ in range(self.lm.cfg.n_repeats) for blk in self.lm.blocks]   # the unique blocks are run n_repeats times
        nf = self.n_nograd
        if nf:
            with torch.no_grad():
                h = self.lm.tok_emb(x)
                for blk in passes[:nf]:
                    h = blk(h)
        else:
            h = self.lm.tok_emb(x)
        for blk in passes[nf:]:
            if self.use_ckpt and self.training and torch.is_grad_enabled():
                h = checkpoint(blk, h, use_reentrant=False)
            else:
                h = blk(h)
        return self.lm.ln_f(h)

    def forward(self, X, lens, cnt):
        h = self.features(X)                                                   # (N, T, d)
        N, T, _ = h.shape
        lens = lens.to(h.device)
        m = (torch.arange(T, device=h.device)[None, :] < lens[:, None]).to(h.dtype)
        mean = (h * m.unsqueeze(-1)).sum(1) / lens.unsqueeze(-1).to(h.dtype)
        lastv = h[torch.arange(N, device=h.device), lens - 1]
        v = self.proj(torch.cat([mean, lastv], -1))                            # (N, proj): one vector per chunk
        D, Kmax = cnt.numel(), int(cnt.max())
        pad = v.new_zeros(D, Kmax, v.size(-1))
        mask = torch.zeros(D, Kmax, dtype=torch.bool, device=v.device)
        pos = 0
        for i, c in enumerate(cnt.tolist()):
            pad[i, :c] = v[pos:pos + c]; mask[i, :c] = True; pos += c
        if self.agg == "mean":
            z = (pad * mask.unsqueeze(-1).to(pad.dtype)).sum(1) / mask.sum(1, keepdim=True).to(pad.dtype)
        elif self.agg == "attn":
            s = self.att(pad).squeeze(-1).float().masked_fill(~mask, float("-inf"))
            w = F.softmax(s, dim=1).to(pad.dtype)
            z = (w.unsqueeze(-1) * pad).sum(1)
        else:
            packed = nn.utils.rnn.pack_padded_sequence(pad, cnt, batch_first=True, enforce_sorted=False)
            o, _ = self.rnn(packed)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=Kmax)
            z = (o * mask.unsqueeze(-1).to(o.dtype)).sum(1) / mask.sum(1, keepdim=True).to(o.dtype)
        return self.out(self.drop(z))                                          # (D, 2)


def build_model(method, agg="attn", r=16, alpha=32, lorma_r=8, lorma_alpha=8, bottleneck=64):
    lm = load_lm()
    if method not in ("full", "partial"):
        for p in lm.parameters(): p.requires_grad = False
    if method == "lora":      apply_lora(lm.blocks, r, alpha, rs=False)
    elif method == "lorma":   apply_lorma(lm.blocks, ("q_proj", "v_proj"), lorma_r, lorma_alpha)
    elif method == "adapters": apply_adapters(lm, bottleneck)
    elif method == "partial":                       # layer-wise: freeze embeddings + lower half of the 4 unique blocks (frozen in both passes)
        for p in lm.tok_emb.parameters(): p.requires_grad = False
        for b in lm.blocks[: len(lm.blocks) // 2]:
            for p in b.parameters(): p.requires_grad = False
    elif method != "full":
        raise ValueError(method)
    m = CJModel(lm, agg)
    m.out.bias.data = torch.tensor(PRIOR_LOGIT, dtype=torch.float32)      # start from the label prior (rare statutes start near 0)
    return m.to(dev)


def make_student(teacher):
    """Smaller student: keep unique blocks 0 and 2 (half the block parameters) and still run them n_repeats times
    (depth 8 -> 4). Shared embeddings, initialised from the teacher incl. its trained head.
    The teacher must be the plain full-fine-tuned model."""
    st = copy.deepcopy(teacher)
    keep = list(range(0, len(st.lm.blocks), 2))
    st.lm.blocks = nn.ModuleList([st.lm.blocks[i] for i in keep])
    st.lm.cfg.n_unique_blocks = len(keep)
    for p in st.parameters(): p.requires_grad = True
    return st

## 8. Inference and evaluation

In [ ]:
@torch.no_grad()
def predict_probs(model, docs, select=None, K=None, bs=1):
    """-> numpy array (len(docs), NL) of sigmoid probabilities, aligned with `docs`."""
    model.eval()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["chunks"]))
    probs = np.zeros((len(docs), NL), dtype=np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        X, lens, cnt, _ = collate([docs[j] for j in idx], select, K)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            lg = model(X.to(dev), lens, cnt)
        probs[idx] = torch.sigmoid(lg.float()).cpu().numpy()
    return probs

def evaluate(model, docs, select=None, K=None, verbose=True):
    probs = predict_probs(model, docs, select, K)
    Y = np.stack([d["y"] for d in docs])
    t, f_best = best_global_threshold(probs, Y)
    r5 = multilabel_scores(Y, decode(probs, 0.5)); rb = multilabel_scores(Y, decode(probs, t))
    if verbose:
        print(f"   dev macro-F1 @0.5 {r5['macro_f1']:.4f} | @best global thr {t:.2f}: macro {rb['macro_f1']:.4f} "
              f"micro {rb['micro_f1']:.4f} subset-acc {rb['subset_acc']:.4f} | labels/doc predicted {rb['avg_pred']:.2f}")
    return dict(macro_f1=rb["macro_f1"], thr=t, probs=probs, **{k: v for k, v in rb.items() if k != "macro_f1"})

## 9. Time-budgeted multi-label training loop (all methods)

In [ ]:
def train_fast(method, train_docs, dev_docs, minutes=40, K=None, bs=1, accum=8, lr_bb=3e-5, lr_head=1e-3, agg="attn",
               teacher=None, kd_T=2.0, kd_alpha=0.5, loss_type="wbce", pw_power=0.5, pw_max=30.0, select=None,
               top_blocks=None, ckpt=True, seed=0, eval_every=5, eval_n=200, init_model=None):
    """Time-budgeted multi-label trainer: trains for `minutes` of wall-clock time, then evaluates on the whole dev set.
    method: full | partial | lora | lorma | adapters | kd.
    loss_type: bce | wbce | focal | asl.   pw_power: positive-weight exponent for wbce ((neg/pos)**power, capped at pw_max).
    top_blocks=n -> only the top n blocks (+ head) train; the embedding and lower blocks run under no_grad (n=0: head only).
    init_model   -> continue training a model already in memory (use lower lrs).   ckpt=True -> gradient checkpointing.
    eval_every   -> minutes between dev checks on the first eval_n dev docs (None = only the final evaluation); not charged to the budget.
    KD: loss = alpha * BCE(hard labels) + (1-alpha) * T^2 * BCE(student/T, sigmoid(teacher/T))."""
    random.seed(seed); torch.manual_seed(seed)
    if init_model is not None:
        model = init_model
    elif method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method, agg)
    model.use_ckpt = ckpt
    if top_blocks is not None and method in ("full", "partial", "kd"):
        model.set_top_blocks(top_blocks)
    print(f"[{method}/{agg}/{loss_type}/{select or SELECT}/K={K or K_CHUNKS}/top={top_blocks}] params {n_params(model)/1e6:.1f}M | "
          f"trainable {n_params(model, True)/1e6:.2f}M | budget {minutes} min")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    n_tr = len(train_docs); pos = np.stack([d["y"] for d in train_docs]).sum(0)
    pos_w = torch.tensor(np.minimum(((n_tr - pos) / (pos + 1.0)) ** pw_power, pw_max), dtype=torch.float32, device=dev)

    budget, t0, mb, seen, run, nrun, last_log = minutes * 60.0, time.time(), 0, 0, 0.0, 0, time.time()
    last_eval = time.time()
    dev_sub = dev_docs[:eval_n] if (dev_docs and eval_every) else None
    done = False
    while not done:
        order = list(range(len(train_docs))); random.shuffle(order)
        for i in range(0, len(order), bs):
            el = time.time() - t0
            if el >= budget:
                done = True; break
            model.train()
            X, lens, c, Y = collate([train_docs[j] for j in order[i:i + bs]], select, K)
            X, Y = X.to(dev), Y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X, lens, c)
            lg = lg.float()
            loss = multilabel_loss(lg, Y, loss_type, pos_w)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                    tl = teacher(X, lens, c).float()
                kd = F.binary_cross_entropy_with_logits(lg / kd_T, torch.sigmoid(tl / kd_T), reduction="none").sum(1).mean() * kd_T ** 2
                loss = kd_alpha * loss + (1 - kd_alpha) * kd
            scaler.scale(loss / accum).backward()
            run += loss.item(); nrun += 1; seen += len(Y); mb += 1
            if mb % accum == 0:
                prog = el / budget
                mult = prog / 0.05 if prog < 0.05 else 0.5 * (1 + math.cos(math.pi * (prog - 0.05) / 0.95))
                for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * max(mult, 0.02)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if dev_sub and time.time() - last_eval > eval_every * 60:
                te = time.time()
                print(f"   -- dev check at {(te-t0)/60:.1f} min ({len(dev_sub)} dev docs):", end="")
                evaluate(model, dev_sub, select, K)
                t0 += time.time() - te; last_eval = time.time(); last_log = time.time()
            if time.time() - last_log > 300:
                print(f"   {(time.time()-t0)/60:5.1f} min | docs seen {seen} ({seen/len(train_docs):.2f} epochs) | "
                      f"{seen/(time.time()-t0):.2f} docs/s | loss {run/max(1,nrun):.4f}")
                run, nrun, last_log = 0.0, 0, time.time()
    print(f"trained {(time.time()-t0)/60:.1f} min: {seen} docs = {seen/len(train_docs):.2f} epochs ({seen/(time.time()-t0):.2f} docs/s)")
    sc = evaluate(model, dev_docs, select, K) if dev_docs else {"macro_f1": float("nan")}
    return model, sc["macro_f1"]

## 10. Submission writer
`prediction.json` (a list of `{"id", "pred": [statute ids]}`) goes at the **root** of the zip. `run_and_submit` tunes the threshold on dev, writes the zip(s) and saves probabilities for ensembling. Upload each zip to Codabench.

In [ ]:
def write_zip(probs, docs, tag, thr):
    """prediction.json = [{"id": ..., "pred": [statute ids]}, ...] at the root of the zip."""
    Pm = decode(probs, thr)
    sub = [{"id": d["id"], "pred": [int(LABELS[j]) for j in np.nonzero(Pm[i])[0]]} for i, d in enumerate(docs)]
    assert len(sub) == len(docs) and all(len(set(s["pred"])) == len(s["pred"]) and all(isinstance(x, int) for x in s["pred"]) for s in sub)
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(sub, f)
    name = f"{OUT}/submission_p5_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print(" ", name, "| docs:", len(sub), "| statutes/doc:", round(float(np.mean([len(s["pred"]) for s in sub])), 2))
    return name

def run_and_submit(model, tag, per_label=True, select=None, K=None):
    """Dev probabilities -> best global threshold -> submission zip.  With per_label=True a second zip with per-statute thresholds
    is written, and a split-half check tells you whether per-statute thresholds really beat the global one.
    Dev and test probabilities are also saved as .npy (for ensembling later)."""
    dp = predict_probs(model, dev_docs, select, K); Yd = np.stack([d["y"] for d in dev_docs])
    t, f = best_global_threshold(dp, Yd); r5 = multilabel_scores(Yd, decode(dp, 0.5)); rb = multilabel_scores(Yd, decode(dp, t))
    print(f"[{tag}] dev macro-F1 @0.5 {r5['macro_f1']:.4f} | global thr {t:.2f}: macro {f:.4f} micro {rb['micro_f1']:.4f} "
          f"subset-acc {rb['subset_acc']:.4f} (macro incl. absent statutes {rb['macro_f1_all']:.4f})")
    tp_ = predict_probs(model, test_docs, select, K)
    np.save(f"{OUT}/probs_dev_{tag}.npy", dp); np.save(f"{OUT}/probs_test_{tag}.npy", tp_)
    write_zip(tp_, test_docs, tag, t)
    if per_label:
        pl, gl = per_label_cv(dp, Yd, t)
        print(f"   split-half check: per-statute thresholds {pl:.4f} vs global {gl:.4f}  ->  "
              f"{'per-statute helps' if pl > gl else 'global is as good - prefer the plain zip'}")
        write_zip(tp_, test_docs, tag + "_perlabel", per_label_thresholds(dp, Yd, t))
    return dict(thr=t, macro_f1=f)

results = {}

## 11. Experiment 1 - full fine-tuning (also the KD teacher)
All blocks trainable (backbone lr 3e-5, head lr 1e-3) for a fixed 120-minute budget. Keep this model in memory for distillation. If you run out of GPU memory, lower `bs` (raise `accum`) or pass `ckpt=True`.

In [ ]:
m_full, results["full"] = train_fast("full", train_docs, dev_docs, minutes=120)
torch.save(m_full.state_dict(), f"{OUT}/p5_lightweight_full.pt")
run_and_submit(m_full, "lightweight_full")

## 12. Experiment 2 - partial (layer-wise) fine-tuning
Embedding and the first 4 of the 8 block passes run under no_grad (no backward through them); the lower half of the unique blocks is also frozen; the rest and the head train.

In [ ]:
m_part, results["partial"] = train_fast("partial", train_docs, dev_docs, minutes=45, top_blocks=4)
run_and_submit(m_part, "lightweight_partial")

## 13. Experiment 3 - LoRA
Frozen backbone, rank-16 adapters on every Linear layer (attention and SwiGLU).

In [ ]:
m_lora, results["lora"] = train_fast("lora", train_docs, dev_docs, minutes=45, lr_bb=5e-4)
run_and_submit(m_lora, "lightweight_lora")

## 14. Experiment 4 - LoRMA+ (compulsory)
Multiplicative low-rank adaptation (Bihany, Patel, Modi 2025) of the query/value projections of the attention in each unique block.

In [ ]:
m_lorma, results["lorma"] = train_fast("lorma", train_docs, dev_docs, minutes=45, lr_bb=3e-4)
run_and_submit(m_lorma, "lightweight_lorma")

## 15. Experiment 5 - bottleneck adapters
A small zero-initialised down/up adapter after every unique block; the backbone stays frozen.

In [ ]:
m_adapt, results["adapters"] = train_fast("adapters", train_docs, dev_docs, minutes=45, lr_bb=1e-3)
run_and_submit(m_adapt, "lightweight_adapters")

## 16. Experiment 6 - knowledge distillation
Teacher = experiment 1. Student = teacher keeping unique blocks 0 and 2 (half the block parameters; effective depth 8 -> 4), initialised from the teacher, trained on hard labels + temperature-scaled teacher probabilities (per-statute sigmoid soft targets). Compare its parameter count, speed and F1 with the teacher.

In [ ]:
m_kd, results["kd"] = train_fast("kd", train_docs, dev_docs, minutes=45, teacher=m_full, lr_bb=5e-5, top_blocks=4)
run_and_submit(m_kd, "lightweight_kd")
print("params: teacher", round(n_params(m_full)/1e6, 1), "M | student", round(n_params(m_kd)/1e6, 1), "M")

## 17. Loss-function ablation (optional; 30-min runs training the last 4 block passes, compare with experiment 2 = class-weighted BCE)
Plain BCE vs focal vs asymmetric loss on the rare-statute problem; only the loss changes.

In [ ]:
for lt in ["bce", "focal", "asl"]:
    m_tmp, results[f"loss_{lt}"] = train_fast("partial", train_docs, dev_docs, minutes=30, top_blocks=4, loss_type=lt)
    run_and_submit(m_tmp, f"lightweight_loss_{lt}", per_label=False)
    del m_tmp; torch.cuda.empty_cache()

## 18. Long-document ablations (optional)
Which part of a document carries the statute signal? `head`, `tail`, `spread` vs the default `headtail`; then the aggregator (`mean`, `gru` vs the default `attn`). Report what the dev scores say about sections.

In [ ]:
for sel in ["head", "tail", "spread"]:
    m_tmp, results[f"sel_{sel}"] = train_fast("partial", train_docs, dev_docs, minutes=30, top_blocks=4, select=sel)
    del m_tmp; torch.cuda.empty_cache()
for ag in ["mean", "gru"]:
    m_tmp, results[f"agg_{ag}"] = train_fast("partial", train_docs, dev_docs, minutes=30, top_blocks=4, agg=ag)
    del m_tmp; torch.cuda.empty_cache()

## 19. Summary and rare-statute analysis

In [ ]:
print("dev macro-F1 (best global threshold):")
for k, v in results.items():
    print(f"  {k:20s} {v:.4f}")

# --- rare statutes: F1 by how often the statute appears in train (uses m_full) ---
dp = predict_probs(m_full, dev_docs); Yd = np.stack([d["y"] for d in dev_docs]); t, _ = best_global_threshold(dp, Yd)
f1, tp, fp, fn = label_f1(Yd, decode(dp, t))
print(f"\nglobal threshold {t:.2f}")
for lo, hi in [(0, 1), (1, 5), (5, 20), (20, 100), (100, 10**9)]:
    sel = (freq >= lo) & (freq < hi) & (Yd.sum(0) > 0)
    if sel.any(): print(f"  statutes with {lo}-{hi-1 if hi < 10**9 else '+'} train docs: {int(sel.sum()):4d} in dev | mean F1 {f1[sel].mean():.3f}")
miss = np.argsort(-fn)[:10]
print("\nmost-missed statute ids (false negatives):", [(LABELS[j], int(fn[j])) for j in miss])
spur = np.argsort(-fp)[:10]
print("most over-predicted statute ids (false positives):", [(LABELS[j], int(fp[j])) for j in spur])

## 20. Ensemble (optional, usually the biggest single gain)
Averages saved probabilities. The `.npy` files from **other notebooks** (decoder, encoder-decoder, lightweight) can be copied into `OUT`: all notebooks share the same label order and document order, so they average cleanly. Tune the threshold on the averaged dev probabilities.

In [ ]:
tags = ["lightweight_full", "lightweight_partial", "lightweight_lora", "lightweight_lorma", "lightweight_adapters"]   # add tags from other notebooks
weights = None                                                                                # e.g. [2, 1, 1, 1, 1]
Yd = np.stack([d["y"] for d in dev_docs])
D = [np.load(f"{OUT}/probs_dev_{t}.npy") for t in tags]; T = [np.load(f"{OUT}/probs_test_{t}.npy") for t in tags]
ens_d = np.average(D, axis=0, weights=weights); ens_t = np.average(T, axis=0, weights=weights)
for t_, d_ in zip(tags, D):
    print(f"  {t_:22s} {best_global_threshold(d_, Yd)[1]:.4f}")
t, f = best_global_threshold(ens_d, Yd)
print(f"ensemble: global thr {t:.2f} -> dev macro-F1 {f:.4f}")
write_zip(ens_t, test_docs, "ensemble", t)
pl, gl = per_label_cv(ens_d, Yd, t); print(f"split-half check: per-statute {pl:.4f} vs global {gl:.4f}")

## 21. (Optional) retrain on train + dev
Once you know the best method and budget, retrain on all labelled data and submit with the threshold found above (no dev set remains to tune on, so reuse it).
```python
m_final, _ = train_fast("full", train_docs + dev_docs, None, minutes=<same as the best run>)
tp_ = predict_probs(m_final, test_docs); write_zip(tp_, test_docs, "hybrid_final", <threshold from dev>)
```